# Heart — Held-out logistic-regression evaluation

The pre-specified model is L2 logistic regression with 10-fold cross-validation. Accuracy and F1 are reported for the 5-fold and 10-fold variants, with ROC-AUC also reported for the 10-fold variant. These comparisons do not determine model selection.

Evaluation uses the original stratified 50/50 split and the same held-out observations as replication, rather than an additional independent cohort.

Standardisation is fitted on the training partition before internal cross-validation; it is not refitted within each cross-validation fold.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV

file_path = "../../data/heart/processed.cleveland.data"

# column names -> defined manually
column_names = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs',
    'restecg', 'thalach', 'exang', 'oldpeak',
    'slope', 'ca', 'thal', 'target'
]

df = pd.read_csv(file_path, names=column_names)

# replace missing values '?'
df.replace('?', np.nan, inplace=True)

# conversion numeric columns
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# drop missing values
df = df.dropna()

# conversion to binary classification
df['target'] = df['target'].apply(lambda x: 1 if x > 0 else 0)

X = df.drop('target', axis=1)
y = df['target']

feature_names = X.columns.tolist()

# The source paper uses equal training and test subsets. Split before
# learning preprocessing parameters to prevent test-data leakage.
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y,
    test_size=0.50,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train = pd.DataFrame(
    scaler.fit_transform(X_train_raw),
    columns=feature_names,
    index=X_train_raw.index,
)
X_test = pd.DataFrame(
    scaler.transform(X_test_raw),
    columns=feature_names,
    index=X_test_raw.index,
)

In [2]:
#test check
print("Unique values in y_train:", np.unique(y_train))
print("Unique values in y_test:", np.unique(y_test))
print("Missing values in y_train:", y_train.isnull().sum())
print("Missing values in y_test:", y_test.isnull().sum())

Unique values in y_train: [0 1]
Unique values in y_test: [0 1]
Missing values in y_train: 0
Missing values in y_test: 0


In [3]:
from sklearn.metrics import accuracy_score, f1_score, roc_curve, auc


## Existing 5-fold and 10-fold LR evaluation

In [4]:
# LR with 5-fold and 10-fold CV
lr_5fold = LogisticRegressionCV(Cs=10, cv=5, penalty='l2', solver='liblinear', max_iter=1000, random_state=42)
lr_10fold = LogisticRegressionCV(Cs=10, cv=10, penalty='l2', solver='liblinear', max_iter=1000, random_state=42)

lr_5fold.fit(X_train, y_train)
lr_10fold.fit(X_train, y_train)

y_pred_lr_5fold = lr_5fold.predict(X_test)
y_pred_lr_10fold = lr_10fold.predict(X_test)

print("Logistic Regression (5-fold CV) Test Accuracy:", accuracy_score(y_test, y_pred_lr_5fold))
print("Logistic Regression (5-fold CV) F1 Score:", f1_score(y_test, y_pred_lr_5fold))
print("Logistic Regression (10-fold CV) Test Accuracy:", accuracy_score(y_test, y_pred_lr_10fold))
print("Logistic Regression (10-fold CV) F1 Score:", f1_score(y_test, y_pred_lr_10fold))

Logistic Regression (5-fold CV) Test Accuracy: 0.8523489932885906
Logistic Regression (5-fold CV) F1 Score: 0.8405797101449275
Logistic Regression (10-fold CV) Test Accuracy: 0.8523489932885906
Logistic Regression (10-fold CV) F1 Score: 0.8405797101449275


## Pre-specified 10-fold LR: held-out ROC-AUC

In [5]:
y_score_lr = lr_10fold.predict_proba(X_test)[:, 1]

fpr_lr, tpr_lr, _ = roc_curve(
    y_test,
    y_score_lr
)

auc_lr = auc(
    fpr_lr,
    tpr_lr
)

print(
    f"Logistic Regression AUC: {auc_lr:.2f}"
)

Logistic Regression AUC: 0.92


In [6]:
# Pre-specified replicated variant; held-out ranking is not used for selection.
final_model = lr_10fold
selected_model_name = "Logistic Regression (10-fold CV)"
best_model = final_model


## Summary of the existing comparison

In [7]:
validation_df = pd.DataFrame({
    "Model": ["Logistic Regression (5-fold CV)", selected_model_name],
    "Accuracy": [accuracy_score(y_test, y_pred_lr_5fold),
                 accuracy_score(y_test, y_pred_lr_10fold)],
    "F1": [f1_score(y_test, y_pred_lr_5fold),
           f1_score(y_test, y_pred_lr_10fold)],
})
display(validation_df)
print("Pre-specified 10-fold LR ROC-AUC:", auc_lr)


,Model,Accuracy,F1
0,Logistic Regression (5-fold CV),0.852349,0.84058
1,Logistic Regression (10-fold CV),0.852349,0.84058


Pre-specified 10-fold LR ROC-AUC: 0.9152173913043478
